# Machine and operation classification

Run these cells from the repository root after installing `requirements.txt`.
The baseline uses healthy recordings, 4,000-sample windows, and separate Random
Forest classifiers for machine and operation identification. Entire recordings
stay in one split.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GroupShuffleSplit

from utils import data_loader_utils

WINDOW_SIZE = 4000
SAMPLE_RATE = 2000
RANDOM_STATE = 42

data_path = Path("./data").resolve()
file_list = sorted(data_loader_utils.find_all_h5s_in_dir(data_path))
good_file_list = [path for path in file_list if Path(path).parent.name == "good"]

if not good_file_list:
    raise ValueError("No healthy recordings found. Check the data directory.")

print("Healthy recordings:", len(good_file_list))

## Check one window

At 2 kHz, 4,000 samples span two seconds. X, Y, Z, and their combined magnitude
each contribute 13 time-domain and 12 frequency-domain features shared with the
anomaly detection branch, for 100 inputs. The Welch spectrum uses a Hann
window and 1,024-sample segments. Standard deviation uses the sample estimate.
Skew and excess kurtosis are set to zero for constant signals, where they are
undefined.

In [ ]:
file_df = data_loader_utils.format_file(good_file_list[0])
first_window = file_df.loc[
    file_df["window"] == 0, ["X", "Y", "Z"]
].to_numpy()

if len(first_window) != WINDOW_SIZE:
    raise ValueError("The first recording has no complete two-second window.")

first_features = data_loader_utils.extract_window_features(first_window, fs=SAMPLE_RATE)
print("Window shape:", first_window.shape)
pd.DataFrame([first_features])

## Build the feature table

Each row describes one complete window. Incomplete trailing windows are skipped.
Source files and labels accompany the features for splitting and evaluation.

In [ ]:
rows = []
skipped_windows = 0

for file_path in good_file_list:
    path = Path(file_path)
    recording = data_loader_utils.format_file(path)

    for window_id, window in recording.groupby("window", sort=True):
        if len(window) != WINDOW_SIZE:
            skipped_windows += 1
            continue

        row = data_loader_utils.extract_window_features(
            window[["X", "Y", "Z"]].to_numpy(), fs=SAMPLE_RATE
        )
        row.update({
            "source_file": path.resolve().relative_to(data_path).as_posix(),
            "window": int(window_id),
            "machine": int(window["machine"].iloc[0]),
            "process": int(window["process"].iloc[0]),
        })
        rows.append(row)

if not rows:
    raise ValueError("No complete windows found. Check the data directory.")

features_df = pd.DataFrame(rows)

print("Windows:", len(features_df))
print("Recordings:", features_df["source_file"].nunique())
print("Incomplete windows skipped:", skipped_windows)
features_df.head()

## Split by recording

Hold out 20% of source recordings with a fixed random seed. Machine, process,
source-file, and window metadata never enter the model inputs. The percentage
applies to recordings, so the fraction of held-out windows can differ.

In [ ]:
feature_columns = list(first_features)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(
    splitter.split(features_df, groups=features_df["source_file"])
)

train = features_df.iloc[train_idx]
test = features_df.iloc[test_idx]

assert set(train["source_file"]).isdisjoint(test["source_file"])

print("Training recordings:", train["source_file"].nunique())
print("Test recordings:", test["source_file"].nunique())
print("Training windows:", len(train))
print("Test windows:", len(test))

for target in ("machine", "process"):
    missing_classes = set(features_df[target]) - set(train[target])
    if missing_classes:
        raise ValueError(
            f"{target}: classes absent from training: {missing_classes}. "
            "Check recording counts before defining the split."
        )

    coverage = pd.concat(
        [train[target].value_counts(), test[target].value_counts()],
        axis=1,
        keys=["train_windows", "test_windows"],
    ).fillna(0).astype(int).sort_index()
    print(f"\n{target} class coverage:")
    print(coverage)

## Train and evaluate the classifiers

Start with the forest settings from the highest-PR-AUC Isolation Forest trial
in [the anomaly notebook](https://github.com/Mageto-Nyakoni/cnc-machine-fault-detection/blob/47955474ec2e8a6a7ad1fd757859f8851fd49fcc/anomaly_detection.ipynb).
Each Random Forest has 300 trees and draws up to 1,024 training windows per tree
with replacement. It considers 70% of the features at each split. These are
starting values for classification.

Both classifiers use the same feature table and recording split. Reports
describe predictions for windows from held-out healthy recordings.

In [ ]:
models = {}

for target in ("machine", "process"):
    model = RandomForestClassifier(
        n_estimators=300,
        bootstrap=True,
        max_samples=min(1024, len(train)),
        max_features=0.7,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(train[feature_columns], train[target])
    predictions = model.predict(test[feature_columns])
    models[target] = model

    print(f"\nTarget: {target}")
    print(classification_report(
        test[target],
        predictions,
        labels=sorted(features_df[target].unique()),
        digits=3,
        zero_division=0,
    ))